# EDA — Camiones importados (tabla NORMALIZADA con LLM)

Fuente: `camiones_8704229000_normalizado.xlsx` (hoja `normalizado_llm`). Veritrade · Perú Importaciones · partida `8704229000` · Ene-2023 a Abr-2026 · 12.417 unidades.

Marca/modelo y los 4 enums (tracción, combustible, clasificación, caja) están **normalizados contra el vocabulario controlado** (`ejemplo.xlsx` + `data/vocab_extra.json`).

> ⚠️ **Contenido generado/asistido por IA** (DeepSeek): `marca_norm`, `modelo_match` y `*_norm`. `modelo_flag` indica la confianza: `ok` (match exacto), `alias` (mapeo curado inferido — validar), `low`/`nomatch` (en revisión). Validar con experto antes de decisiones.


In [ ]:
import os
# correr desde la raíz del repo aunque el notebook esté en notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 150)
plt.rcParams["figure.figsize"]=(10,4.5); plt.rcParams["axes.grid"]=True

df = pd.read_excel("outputs/veritrade_PE_I_8704229000_normalizado.xlsx", sheet_name="normalizado_llm")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
obj = df.select_dtypes("object").columns
df[obj] = df[obj].apply(lambda s: s.str.strip())
print(df.shape); df[["marca_norm","modelo_match","modelo_flag","combustible_norm","caja_norm","traccion_norm","clasificacion_norm"]].head()


## 1. Calidad de la normalización


In [ ]:
print("modelo_flag:")
print(df["modelo_flag"].value_counts(dropna=False).to_string())
resueltos = df["modelo_flag"].isin(["ok","alias","low"]).mean()
print(f"\nmodelo resuelto (ok+alias+low): {resueltos:.1%}")
print(f"marca en vocabulario: {df['marca_in_vocab'].mean():.1%}")
for c in ["traccion","combustible","clasificacion","caja"]:
    print(f"{c}_norm fill: {df[c+'_norm'].notna().mean():.1%}  | válido: {df[c+'_valido'].mean():.1%}")
ax=df["modelo_flag"].value_counts().plot(kind="bar", color="#4C78A8", title="modelo_flag", rot=0); plt.tight_layout(); plt.show()


## 2. Marcas canónicas — participación de mercado


In [ ]:
top = df["marca_norm"].value_counts().head(15)
ax=top.iloc[::-1].plot(kind="barh", color="#54A24B", title="Top 15 marcas (normalizadas) por unidades"); ax.set_xlabel("unidades"); plt.tight_layout(); plt.show()
(df["marca_norm"].value_counts(normalize=True).head(10)*100).round(1).to_frame("market_share_%")


## 3. Modelos canónicos


In [ ]:
print("=== Top 15 modelos (modelo_match) global ===")
print(df["modelo_match"].value_counts().head(15).to_string())
print("\n=== Top 3 modelos por marca (top 6 marcas) ===")
top6 = df["marca_norm"].value_counts().head(6).index
(df[df["marca_norm"].isin(top6)].dropna(subset=["modelo_match"])
   .groupby(["marca_norm","modelo_match"]).size().sort_values(ascending=False)
   .groupby(level=0).head(3)).to_frame("unidades")


## 4. Especificaciones normalizadas


In [ ]:
fig, ax = plt.subplots(2,2, figsize=(13,8))
df["combustible_norm"].value_counts().plot(kind="bar", ax=ax[0,0], title="Combustible", rot=0)
df["caja_norm"].value_counts().plot(kind="bar", ax=ax[0,1], title="Caja (transmisión)", rot=0)
df["traccion_norm"].value_counts().plot(kind="bar", ax=ax[1,0], title="Tracción", rot=0)
df["clasificacion_norm"].value_counts().plot(kind="bar", ax=ax[1,1], title="Clasificación", rot=0)
plt.tight_layout(); plt.show()


## 5. Cruce marca × clasificación


In [ ]:
ct = pd.crosstab(df["marca_norm"], df["clasificacion_norm"])
ct["total"]=ct.sum(axis=1)
ct.sort_values("total", ascending=False).head(12)


## 6. Valor comercial por marca


In [ ]:
print("CIF total (USD): {:,.0f}".format(df["cif_usd"].sum()))
g=(df.groupby("marca_norm").agg(unidades=("cif_usd","size"), cif_total=("cif_usd","sum"), cif_prom_unit=("cif_usd","mean"))
     .sort_values("unidades", ascending=False).head(12))
g["cif_total"]=g["cif_total"].round(0); g["cif_prom_unit"]=g["cif_prom_unit"].round(0)
display(g)
ax=g.sort_values("cif_prom_unit")["cif_prom_unit"].plot(kind="barh", color="#B279A2", title="CIF promedio por unidad — top marcas (USD)"); plt.tight_layout(); plt.show()


## 7. Evolución temporal y mix de marcas


In [ ]:
por_mes=df.set_index("fecha").resample("MS").size()
ax=por_mes.plot(marker="o", color="#4C78A8", title="Unidades por mes"); ax.set_xlabel(""); plt.tight_layout(); plt.show()
pivot=(df.assign(anio=df["fecha"].dt.year).pivot_table(index="marca_norm", columns="anio", values="vin", aggfunc="count", fill_value=0))
pivot["total"]=pivot.sum(axis=1)
pivot.sort_values("total", ascending=False).head(12)


## 8. Pendiente de validación (calidad)


In [ ]:
print("Filas por estado de modelo a revisar:")
print(df[df["modelo_flag"].isin(["alias","low","nomatch"])]["modelo_flag"].value_counts().to_string())
print("\n=== Marcas nuevas (fuera del ejemplo) ===")
try:
    nv=pd.read_excel("outputs/veritrade_PE_I_8704229000_normalizado.xlsx", sheet_name="_vocab_nuevo")
    print(nv[["marca_norm","unidades","sugerencia"]].to_string(index=False))
except Exception as e:
    print("(_vocab_nuevo no disponible:", e, ")")
print("\n=== Ejemplo de filas 'alias' (mapeo inferido a validar) ===")
display(df[df["modelo_flag"]=="alias"][["marca_norm","modelo_raw_llm","modelo_match"]].drop_duplicates().head(10))


## Cierre

- Tabla normalizada lista para análisis de mercado por **marca/modelo canónicos** y specs estandarizadas.
- Las filas `alias` (mapeo inferido) y `low`/`nomatch` están en la hoja `_revisar_llm` para validación de experto.
- Re-normalizar tras ajustar el diccionario (`data/vocab_extra.json`) es gratis (cache del LLM).
